# FIRMS — fire-season time-series

Count daily fire pixels and total daily FRP over a fire season for one region, then plot the activity curve. Shows the backend chunking a multi-week window into ≤10-day requests transparently.

In [ ]:
import datetime as dt
import os
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('firms_output')
OUT_DIR.mkdir(exist_ok=True)

# FIRMS needs a free MAP_KEY (https://firms.modaps.eosdis.nasa.gov/api/map_key/).
# Set FIRMS_MAP_KEY in your environment; the live cells below skip cleanly
# (nbval-lax-safe) when it is absent, so the notebook never fails offline.
HAS_KEY = bool(os.environ.get('FIRMS_MAP_KEY'))
print('FIRMS_MAP_KEY set:', HAS_KEY)

## Query a fire-season window

A several-week window is split into ≤10-day chunks internally and returned merged. Use a recent window so the NRT sensor has coverage.

In [ ]:
TODAY = dt.date.today()
START = (TODAY - dt.timedelta(days=30)).strftime('%Y-%m-%d')
END = TODAY.strftime('%Y-%m-%d')

fires = None
if HAS_KEY:
    try:
        fires = EarthLens(
            data_source='firms',
            variables=['VIIRS_SNPP_NRT'],
            start=START,
            end=END,
            aoi=[-124.0, 36.0, -118.0, 42.0],  # Northern California / Pacific NW
            path=str(OUT_DIR),
        ).download(progress_bar=False)
        print('detections:', len(fires))
    except Exception as exc:
        print('skipped live query:', exc)

## Daily fire-pixel count and total FRP

In [ ]:
if fires is not None and len(fires):
    import matplotlib.pyplot as plt

    daily = fires.assign(date=fires['acq_datetime'].dt.date).groupby('date')
    counts = daily.size()
    total_frp = daily['frp'].sum()

    fig, ax1 = plt.subplots(figsize=(9, 4))
    ax1.bar(counts.index, counts.values, color='tab:orange', alpha=0.6)
    ax1.set_ylabel('fire-pixel count', color='tab:orange')
    ax1.set_xlabel('date')
    ax2 = ax1.twinx()
    ax2.plot(total_frp.index, total_frp.values, color='tab:red', marker='o')
    ax2.set_ylabel('total FRP (MW)', color='tab:red')
    ax1.set_title(f'Daily fire activity, {START} to {END}')
    peak = counts.idxmax()
    ax1.annotate(
        f'peak: {peak}',
        xy=(peak, counts.max()),
        xytext=(0.6, 0.85),
        textcoords='axes fraction',
        arrowprops=dict(arrowstyle='->'),
    )
    fig.tight_layout()
    plt.show()